In [ ]:
import os
import h5py
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt
import pandas as pd
from datetime import datetime, timedelta
import pytz
import h5py

# Directorio que contiene los archivos .h5
directory_path = '/data/data4/veronica-scratch-rainier/swarm_august2023/templates-files/template-7second-test'

# Listar todos los archivos .h5 en el directorio
file_list = [f for f in os.listdir(directory_path) if f.endswith('.h5')]

# Función para cargar los datos desde un archivo .h5
def load_h5_file(file_path):
    with h5py.File(file_path, 'r') as f:
        data = np.array(f['Acquisition/Raw[0]/RawData'])
    return data

# Función para diseñar un filtro pasabanda
def design_bandpass_filter(lowcut, highcut, fs, order=5):
    nyquist = 0.5 * fs
    low = lowcut / nyquist
    high = highcut / nyquist
    b, a = butter(order, [low, high], btype='band')
    return b, a

# Función para aplicar el filtro pasabanda a los datos
def apply_bandpass_filter(data, lowcut, highcut, fs, order=5):
    b, a = design_bandpass_filter(lowcut, highcut, fs, order=order)
    y = filtfilt(b, a, data, axis=0)
    return y

# Parámetros del filtro
lowcut = 2.0
highcut = 9.8
fs = 20  # Frecuencia de muestreo en Hz, ajusta esto según tus datos



In [ ]:
# Función para procesar las carpetas
def process_folders(full_path, time_utc, matched_files_with_locations):
    try:
        # Obtener una lista de todas las carpetas en el directorio base
        folders = [folder for folder in os.listdir(full_path) if os.path.isdir(os.path.join(full_path, folder))]
    except Exception as e:
        print(f"Error listing folders in {full_path}: {e}")
        return

    # Inicializar una lista para almacenar los datos concatenados de cada carpeta
    concatenated_data_per_folder = []

    # Iterar sobre las carpetas y cargar los archivos .npy
    for folder in folders:
        folder_path = os.path.join(full_path, folder)
        try:
            npy_files = [np.load(os.path.join(folder_path, file)) for file in os.listdir(folder_path) if file.endswith('.npy')]
            concatenated_data_per_folder.append(np.concatenate(npy_files, axis=0))
        except Exception as e:
            print(f"Error processing folder {folder_path}: {e}")
            continue

    # Calcular MAD para cada carpeta y definir los umbrales
    mads_per_folder = {}
    thresholds_per_folder = {}
    for folder, folder_data in zip(folders, concatenated_data_per_folder):
        try:
            mad = mad_func_shelly(folder_data)
            threshold = 18 * mad
            mads_per_folder[folder] = np.round(mad, decimals=3)
            thresholds_per_folder[folder] = np.round(threshold, decimals=3)
            print(f"MAD para la carpeta {folder}: {mads_per_folder[folder]}")
            print(f"Umbral para la carpeta {folder}: {thresholds_per_folder[folder]}")
        except Exception as e:
            print(f"Error calculating MAD for folder {folder}: {e}")

    # Crear archivos CSV para las detecciones
    output_directory = "csv_results_thresholds"
    if not os.path.exists(output_directory):
        os.makedirs(output_directory)

    detection_times = []

    for folder, folder_data in zip(folders, concatenated_data_per_folder):
        print(f"Processing folder {folder}")
        threshold = thresholds_per_folder[folder]
        indices_above_threshold = np.where(np.abs(folder_data) > threshold)[0]
        diff_indices = np.diff(indices_above_threshold)
        group_changes = np.where(diff_indices > 20)[0]
        detection_groups = np.split(indices_above_threshold, group_changes + 1)

        detection_times_folder = []

        # Buscar coincidencia con la carpeta
        matching_item = next((item for item in matched_files_with_locations if item['original_date'].replace(":", "-").replace(" ", "_") in folder), None)
        if matching_item:
            for group in detection_groups:
                if len(group) > 0:
                    first_detection_time_utc = time_utc[group[0]].strftime('%Y-%m-%d %H:%M:%S')
                    detection_times_folder.append({
                        'Detection Time (UTC)': first_detection_time_utc,
                        'Longitude': matching_item['longitude'],
                        'Latitude': matching_item['latitude'],
                        'Folder': folder
                    })
            print(f"Matching item found for folder {folder}: {matching_item}")
        else:
            print(f"No matching item found for folder {folder}")

        print(f"Detections for folder {folder}: {detection_times_folder}")
        detection_times.extend(detection_times_folder)

        # Guardar en CSV
        df = pd.DataFrame(detection_times_folder)
        df.to_csv(os.path.join(output_directory, f'detections_{folder}.csv'), index=False)

    # Remover duplicados de todos los archivos CSV
    if detection_times:  # Ensure there are detection times to process
        all_detections = pd.DataFrame(detection_times)
        all_detections['Detection Time (UTC)'] = pd.to_datetime(all_detections['Detection Time (UTC)'])
        all_detections = all_detections.sort_values(by='Detection Time (UTC)')

        # Eliminar detecciones que estén dentro de los 10 segundos de cada una
        threshold_seconds = 8
        unique_detections = []
        duplicates_dict = {}
        previous_time = None

        for index, row in all_detections.iterrows():
            detection_time = row['Detection Time (UTC)']
            folder = row['Folder']
            if previous_time is None or (detection_time - previous_time).total_seconds() > threshold_seconds:
                unique_detections.append(detection_time)
                duplicates_dict[detection_time] = [folder]
                previous_time = detection_time
            else:
                duplicates_dict[previous_time].append(folder)

        # Crear DataFrame para las detecciones únicas y sus duplicadas
        unique_detections_data = []
        for unique_time, folders in duplicates_dict.items():
            unique_detections_data.append({
                'Unique Detection Time (UTC)': unique_time,
                'Duplicate Detections (UTC)': ', '.join(folders)
            })

        unique_detections_df = pd.DataFrame(unique_detections_data)

        # Imprimir la primera columna de unique_detections_df
        print("Unique Detection Times (UTC):")
        print(unique_detections_df['Duplicate Detections (UTC)'])

        unique_detections_df.to_csv(os.path.join(output_directory, 'unique_detections.csv'), index=False)

        print(f"Detections saved in {output_directory}")
    else:
        print("No detection times to process.")

# Variables y rutas
base_path = '/data/fast1/veronica-scratch-rainier-downsampling/drive1_ds/'
full_path = '/data/data4/veronica-scratch-rainier/swarm_august2023/results_CC_TMA/CC_10sec-tem_2023-08-27_10.00-2023-08-27_11.00'
output_file_h5 = '/data/data4/veronica-scratch-rainier/swarm_august2023/results_CC_TMA/h5_files_timestamps/timestamps_2023-08-27_10.00.00_2023-08-27_11.00.00.h5'

# Obtener los eventos
events = search(starttime=datetime(2023, 8, 26, 0, 0), 
                endtime=datetime(2023, 8, 31, 0, 0),
                latitude=46.879967,
                longitude=-121.726906,
                maxradius=35/111.32)

event_df = get_summary_data_frame(events)
event_df = event_df.sort_values(by=['time'], ascending=True)
print("Returned %s events" % len(events))

# Encontrar archivos con fechas correspondientes
matched_files_with_locations = find_files_with_dates(event_df, base_path)
print("Matched files with locations:")
print(matched_files_with_locations)

# Convertir timestamps a UTC
time_utc = convert_timestamps_to_utc(output_file_h5)

# Procesar carpetas
process_folders(full_path, time_utc, matched_files_with_locations)